<a href="https://colab.research.google.com/github/Nasim-Amani/grid-failure/blob/main/Prediction_Tasks_Plan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Distribution Network Hierarchy

The electricity distribution path is:

Generation  
↓  
Transmission Network  
↓  
Sub-transmission Substation (پست فوق توزیع)  
↓  
Medium Voltage Feeder (فیدر فشار متوسط) ⭐  
↓  
Distribution Substation (پست توزیع)  
↓  
MV/LV Transformer  
↓  
Low Voltage Feeder (فیدر فشار ضعیف)  
↓  
Customers


## Research Focus: MV Feeder

The main prediction unit in this study is the **Medium Voltage Feeder (MV Feeder)**.

Reason:

- It has a stable identifier (GISCODE فیدر فشار متوسط).
- It has repeated historical outage records.
- It is a meaningful maintenance unit for utilities.
- It provides a balance between network-level and customer-level analysis.


Therefore, the modeling hierarchy is:

Sub-transmission Substation  
↓  
MV Feeder ⭐ (Prediction Asset)  
↓  
Distribution Substation  
↓  
Transformer  
↓  
Customer


The objective is to learn how the **health condition of each MV feeder changes over time** and predict future unplanned failures.

In [ ]:
# ==========================================
# Step 5: Create Monthly Feeder Health Dataset
# ==========================================

import pandas as pd
import numpy as np


# ------------------------------------------
# 1. Read reference dataset
# ------------------------------------------

path = "/content/feeders_clean_outage_reference.csv"

df_reference = pd.read_csv(path)

df_reference["datetime"] = pd.to_datetime(
    df_reference["datetime"]
)


print("Reference data shape:")
print(df_reference.shape)


# ------------------------------------------
# 2. Create monthly time variable
# ------------------------------------------

df_reference["month"] = (
    df_reference["datetime"]
    .dt.to_period("M")
    .astype(str)
)


# ------------------------------------------
# 3. Create monthly feeder aggregation
# ------------------------------------------

monthly_health = (
    df_reference
    .groupby(
        [
            "GISCODE فیدر فشار متوسط",
            "month"
        ]
    )
    .agg(

        # --------------------------
        # Failure behavior
        # --------------------------

        unplanned_failure_count=(
            "failure_target",
            "sum"
        ),

        planned_outage_count=(
            "failure_target",
            lambda x: (x == 0).sum()
        ),

        total_outage_count=(
            "failure_target",
            "count"
        ),


        # --------------------------
        # Unplanned impact
        # --------------------------

        unplanned_duration_minutes=(
            "مدت زمان خاموشی به دقیقه",
            lambda x: x[
                df_reference.loc[x.index, "failure_target"] == 1
            ].sum()
        ),


        unplanned_energy_loss_kwh=(
            "انرژی توزیع نشده (KWH)",
            lambda x: x[
                df_reference.loc[x.index, "failure_target"] == 1
            ].sum()
        ),


        # --------------------------
        # Overall network activity
        # --------------------------

        total_duration_minutes=(
            "مدت زمان خاموشی به دقیقه",
            "sum"
        ),

        total_energy_loss_kwh=(
            "انرژی توزیع نشده (KWH)",
            "sum"
        ),


        # --------------------------
        # Operational stress
        # --------------------------

        switching_operations=(
            "جداکننده",
            "count"
        )

    )
    .reset_index()
)


# ------------------------------------------
# 4. Create complete feeder-month timeline
# ------------------------------------------

feeders = df_reference[
    "GISCODE فیدر فشار متوسط"
].unique()


months = pd.date_range(
    start=df_reference["datetime"].min(),
    end=df_reference["datetime"].max(),
    freq="MS"
).strftime("%Y-%m")


full_index = pd.MultiIndex.from_product(
    [
        feeders,
        months
    ],
    names=[
        "GISCODE فیدر فشار متوسط",
        "month"
    ]
)


monthly_health = (
    monthly_health
    .set_index(
        [
            "GISCODE فیدر فشار متوسط",
            "month"
        ]
    )
    .reindex(full_index)
    .fillna(0)
    .reset_index()
)


# ------------------------------------------
# 5. Add time features
# ------------------------------------------

monthly_health["date"] = pd.to_datetime(
    monthly_health["month"]
)


monthly_health["year"] = (
    monthly_health["date"]
    .dt.year
)


monthly_health["month_number"] = (
    monthly_health["date"]
    .dt.month
)


# ------------------------------------------
# 6. Sort dataset
# ------------------------------------------

monthly_health = monthly_health.sort_values(
    [
        "GISCODE فیدر فشار متوسط",
        "month"
    ]
).reset_index(drop=True)


# ------------------------------------------
# 7. Check result
# ------------------------------------------

print("\nMonthly feeder health shape:")
print(monthly_health.shape)


print("\nFirst rows:")
display(
    monthly_health.head(10)
)


print("\nZero-failure months:")
display(
    monthly_health[
        monthly_health["unplanned_failure_count"] == 0
    ].head()
)

Reference data shape:
(2764, 48)

Monthly feeder health shape:
(456, 13)

First rows:


,GISCODE فیدر فشار متوسط,month,unplanned_failure_count,planned_outage_count,total_outage_count,unplanned_duration_minutes,unplanned_energy_loss_kwh,total_duration_minutes,total_energy_loss_kwh,switching_operations,date,year,month_number
0,112240,2016-04,2.0,6.0,8.0,186.0,57.570903,759.0,1422.960011,8.0,2016-04-01,2016,4
1,112240,2016-05,6.0,2.0,8.0,413.0,583.677268,608.0,868.710453,8.0,2016-05-01,2016,5
2,112240,2016-06,4.0,2.0,6.0,370.0,103.863996,399.0,138.894847,6.0,2016-06-01,2016,6
3,112240,2016-07,5.0,1.0,6.0,512.0,1886.198288,590.0,1918.488050,6.0,2016-07-01,2016,7
4,112240,2016-08,3.0,1.0,4.0,151.0,3727.616551,164.0,3728.140735,4.0,2016-08-01,2016,8
5,112240,2016-09,3.0,0.0,3.0,180.0,71.932276,180.0,71.932276,3.0,2016-09-01,2016,9
6,112240,2016-10,1.0,2.0,3.0,53.0,28.666817,160.0,123.111758,3.0,2016-10-01,2016,10
7,112240,2016-11,5.0,0.0,5.0,240.0,129.683119,240.0,129.683119,5.0,2016-11-01,2016,11
8,112240,2016-12,7.0,2.0,9.0,689.0,3079.335410,757.0,3117.254027,9.0,2016-12-01,2016,12
9,112240,2017-01,1.0,1.0,2.0,62.0,21.332961,87.0,82.676426,1.0,2017-01-01,2017,1



Zero-failure months:


,GISCODE فیدر فشار متوسط,month,unplanned_failure_count,planned_outage_count,total_outage_count,unplanned_duration_minutes,unplanned_energy_loss_kwh,total_duration_minutes,total_energy_loss_kwh,switching_operations,date,year,month_number
10,112240,2017-02,0.0,4.0,4.0,0.0,0.0,183.0,115.022426,4.0,2017-02-01,2017,2
17,112240,2017-09,0.0,3.0,3.0,0.0,0.0,81.0,170.771014,3.0,2017-09-01,2017,9
20,112240,2017-12,0.0,4.0,4.0,0.0,0.0,312.0,765.566427,4.0,2017-12-01,2017,12
48,112240,2020-04,0.0,1.0,1.0,0.0,0.0,10.0,24.537388,1.0,2020-04-01,2020,4
60,112240,2021-04,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,2021-04-01,2021,4


# Prediction Task 1 — Failure Probability (Classification)

## Objective
Predict whether an MV feeder will experience an unplanned outage in the future period.

## Target
Future failure occurrence:

0 → No unplanned outage  
1 → Unplanned outage occurs

## Input
Historical feeder health sequence:

- Previous unplanned failures
- Planned outages
- Outage duration history
- Energy loss history
- Switching activity
- Network features
- Time features

## Model Output
Failure probability:

Example:
Feeder A → 0.85 probability of failure


---

# Prediction Task 2 — Failure Severity (Regression)

## Objective
Estimate the expected impact of a future unplanned outage.

## Target
Future failure severity:

Examples:

- Expected energy not supplied (KWH)
- Expected outage duration (minutes)
- Expected failure count

## Input
Same historical feeder health sequence:

- Previous failure behavior
- Previous impact metrics
- Operational stress indicators
- Network characteristics

## Model Output

Severity prediction:

Example:
Feeder A → Expected energy loss = 1000 KWH


---

# Final Integration

After both models are developed:

Risk Score = Failure Probability × Expected Severity

This risk score will be used for feeder ranking and maintenance optimization.